In [ ]:
# Sanity check: one-epoch dry run to measure time+memory for given config
import time, torch, gc, os, random
import numpy as np
import cv2
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler

# Minimal dataset used for finetune tests
class FinetuneDataset(Dataset):
    def __init__(self, image_paths, noise_prob=0.2, noise_std=1.0, resize=128):
        self.image_paths = image_paths
        self.p = noise_prob
        self.noise_std = noise_std
        self.resize = resize
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.resize, self.resize))
        gt = img.copy()
        # normalize to 0..1
        img_f = img.astype(np.float32) / 255.0
        if np.random.rand() < self.p:
            noisy = img_f + np.random.normal(0, self.noise_std, img_f.shape)
            noisy = np.clip(noisy, 0.0, 1.0)
            noisy = (noisy * 255).astype(np.uint8)
        else:
            noisy = img
        noisy = torch.from_numpy(noisy).float().permute(2,0,1) / 255.0
        gt = torch.from_numpy(gt).float().permute(2,0,1) / 255.0
        return noisy, gt

# Recreate the Autoencoder used in the lab (down/up blocks)
class DownSamplingBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(DownSamplingBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.pool(x)
        return x

class UpSamplingBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(UpSamplingBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.upsample(x)
        return x

class Autoencoder(nn.Module):
    def __init__(self, channels=[64,128,256], input_channels=3, output_channels=3):
        super().__init__()
        self.encoder_blocks = nn.ModuleList()
        in_ch = input_channels
        for ch in channels:
            self.encoder_blocks.append(DownSamplingBlock(in_ch, ch))
            in_ch = ch
        self.decoder_blocks = nn.ModuleList()
        for ch in reversed(channels):
            self.decoder_blocks.append(UpSamplingBlock(in_ch, ch))
            in_ch = ch
        self.final_conv = nn.Conv2d(in_ch, output_channels, kernel_size=3, stride=1, padding=1)
        self.final_act = nn.Sigmoid()
    def forward(self, x):
        for block in self.encoder_blocks:
            x = block(x)
        for block in self.decoder_blocks:
            x = block(x)
        x = self.final_conv(x)
        x = self.final_act(x)
        return x


def sanity_run(batch_size=8, epochs=1, lr=0.001, noise_prob=0.2, noise_std=1.0):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print("Device:", device, torch.cuda.get_device_name(0) if device=='cuda' else "(CPU)")
    torch.backends.cudnn.benchmark = True

    # small dataloaders using existing FinetuneDataset or CustomImageDataset
    if 'files' in globals() and len(files)>0:
        sample_files = files[:1024]  # limit subset to speed up test
    else:
        data_dir = r"c:/Users/IMG_PRO/Desktop/Image-Processing-Course-2025/Lab6_Hyperparameter-Tuning/assets/img_align_celeba"
        sample_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.jpg')][:1024]

    if len(sample_files) == 0:
        raise RuntimeError(f"No sample images found in {data_dir}. Please check the dataset path.")

    ds = FinetuneDataset(sample_files, noise_prob=noise_prob, noise_std=noise_std)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

    model = Autoencoder(channels=[64,128,256], input_channels=3, output_channels=3).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    scaler = GradScaler()

    torch.cuda.empty_cache()
    gc.collect()
    if device=='cuda':
        torch.cuda.reset_peak_memory_stats()
    start = time.time()

    model.train()
    for epoch in range(epochs):
        t0 = time.time()
        for images, gt in loader:
            images, gt = images.to(device, non_blocking=True), gt.to(device, non_blocking=True)
            opt.zero_grad()
            with autocast():
                outputs = model(images)
                loss = loss_fn(outputs, gt)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        t1 = time.time()
        print(f"Epoch {epoch+1} time: {t1-t0:.2f}s")

    total = time.time() - start
    print(f"Total time: {total:.2f}s")
    if device=='cuda':
        peak = torch.cuda.max_memory_allocated() / (1024**3)
        print(f"Peak GPU memory used: {peak:.2f} GB")
    return

# Run the sanity check (adjust batch_size to 2 or 8)
sanity_run(batch_size=8, epochs=1, lr=0.001)


Device: cuda NVIDIA GeForce RTX 4070 Ti SUPER


C:\Users\IMG_PRO\AppData\Local\Temp\ipykernel_30604\3178699614.py:110: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
